# Autonomous Research Agent — Enhanced Edition

A LangGraph ReAct research agent with four callable tools and an upgraded retrieval pipeline.

## What is included

1. **LLM provider options:** Groq (original baseline), Google Gemini, and Groq free models.
2. **Search quality:** query expansion, parallel retrieval, hybrid lexical + semantic ranking, cross-encoder reranking, deduplication, and iterative retrieval.
3. **Four agent tools:** web search, ArXiv academic search, Wikipedia, and YouTube video research.
4. **YouTube researcher:** automatic available-language transcript selection, timestamps, summaries, Q&A, and clear handling when captions are unavailable.

**Security:** Add API keys to Google Colab Secrets. Never paste keys into notebook cells or commit them to GitHub. The notebook reads secrets at runtime.

**LLM provider:** This notebook uses Groq. Create a free-tier API key at https://console.groq.com/keys, then add it to Colab Secrets as `GROQ_API_KEY`. No OpenRouter key is used.


## 1. Install dependencies
Run this cell once in Google Colab. Restart the runtime if Colab requests it.

In [1]:
%pip -q install -U langchain langchain-core langgraph langchain-groq langchain-google-genai langchain-openai langchain-community langchain-tavily youtube-transcript-api arxiv wikipedia rank-bm25 sentence-transformers scikit-learn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


## 2. Configure API keys securely

Required for the default free LLM route:
- `GROQ_API_KEY` — create a key at https://openrouter.ai/settings/keys. You do **not** need to buy credits to use free models; choose the Free account and do not add credits.

Optional:
- `TAVILY_API_KEY` — enables web search for broad research. YouTube transcript analysis does not require it.

Add keys under Google Colab → Secrets (key icon), and enable notebook access. Never paste keys into notebook code or share them.

In [2]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

def load_secret(name, required=False):
    value = None
    if IN_COLAB:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = os.environ.get(name)
    if not value and required:
        value = getpass(f"Enter {name} (hidden input): ").strip()
    if value:
        os.environ[name] = value
    return value

# Groq is the only required LLM provider in this notebook.
for key_name in ["GROQ_API_KEY", "TAVILY_API_KEY"]:
    load_secret(key_name, required=False)

print("Secret lookup complete; values are hidden.")
print({name: bool(os.environ.get(name)) for name in ["GROQ_API_KEY", "TAVILY_API_KEY"]})


Secret lookup complete; values are hidden.
{'GROQ_API_KEY': True, 'TAVILY_API_KEY': True}


## 3. Select and initialize the LLM

This version uses Groq's `openrouter/free` router. It selects an available free chat model automatically, so the notebook does not depend on a single model ID becoming unavailable. Groq's free tier does not require a credit purchase; free requests are rate-limited. See https://openrouter.ai/openrouter/free.

In [3]:
import os
import time
from langchain_groq import ChatGroq

LLM_PROVIDER = "groq"
MODEL_NAME = "openai/gpt-oss-120b"  # Groq-hosted model with tool/function calling support

api_key = load_secret("GROQ_API_KEY", required=True)
if not api_key:
    raise RuntimeError("GROQ_API_KEY is missing. Create a key at https://console.groq.com/keys and add it to Colab Secrets.")

llm = ChatGroq(
    model=MODEL_NAME,
    api_key=api_key,
    temperature=0.2,
    max_tokens=4096,
    timeout=90,
    max_retries=2,
)

def invoke_with_retry(model, prompt, attempts=3, initial_wait=3):
    """Retry transient rate-limit and provider errors with exponential backoff."""
    last_error = None
    for attempt in range(attempts):
        try:
            return model.invoke(prompt)
        except Exception as exc:
            last_error = exc
            message = str(exc).lower()
            temporary = any(token in message for token in ["503", "429", "502", "504", "unavailable", "rate limit", "overloaded", "timeout"])
            if not temporary or attempt == attempts - 1:
                raise
            wait_seconds = initial_wait * (2 ** attempt)
            print(f"Temporary Groq/provider error; retrying in {wait_seconds}s ({attempt + 1}/{attempts})...")
            time.sleep(wait_seconds)
    raise last_error

print("LLM initialized successfully.")
print(f"Provider: Groq | Model: {MODEL_NAME}")
print("API key loaded (value hidden).")


LLM initialized successfully.
Provider: Groq | Model: openai/gpt-oss-120b
API key loaded (value hidden).


## 4. Configure the original research sources

Tavily requires a key. ArXiv and Wikipedia do not require paid API keys. Search calls are wrapped so a temporary source failure does not silently destroy the whole research run.

In [4]:
import os
import re
import json
import time
import hashlib

from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Any

from langchain_tavily import TavilySearch
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper

# Tavily is optional. YouTube analysis works without it.
tavily_api_key = load_secret("TAVILY_API_KEY", required=False)
if tavily_api_key:
    web_search_tool = TavilySearch(
        max_results=6,
        name="web_search",
        tavily_api_key=tavily_api_key
    )
    print("Tavily Web Search: Ready")
else:
    @tool
    def web_search_tool(query: str) -> str:
        """Placeholder used when no Tavily key is configured."""
        return "Web search is disabled because TAVILY_API_KEY is not configured. YouTube, ArXiv and Wikipedia tools remain available."
    print("Tavily Web Search: Disabled (optional; add TAVILY_API_KEY to enable)")

arxiv_tool = ArxivQueryRun(
    api_wrapper=ArxivAPIWrapper(top_k_results=5, doc_content_chars_max=3500),
    name="arxiv_academic_search",
    description="Search academic papers and preprints on ArXiv."
)

wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=3000),
    name="wikipedia_search",
    description="Search Wikipedia for background concepts, definitions, and history."
)

print("ArXiv and Wikipedia tools: Ready")

/tmp/ipykernel_487/215147587.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun


Tavily Web Search: Ready
ArXiv and Wikipedia tools: Ready


## 5. Search quality — query expansion and parallel retrieval

The expansion step creates focused subqueries. The three existing sources are then queried concurrently. This produces a wider evidence pool before ranking.

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

def expand_research_query(question: str, max_queries: int = 4) -> List[str]:
    prompt = f'''
Break this research question into up to {max_queries} distinct, useful search queries.
Cover definitions/background, recent developments, evidence or benchmarks, and limitations
when relevant. Return ONLY a JSON array of strings. Avoid duplicates.
Question: {question}
'''
    try:
        response = llm.invoke([SystemMessage(content="You are a precise search-query planner."),
                               HumanMessage(content=prompt)])
        raw = response.content if isinstance(response.content, str) else str(response.content)
        match = re.search(r"\[.*\]", raw, flags=re.S)
        queries = json.loads(match.group(0)) if match else []
        cleaned = [str(q).strip() for q in queries if str(q).strip()]
        return [question] + cleaned[:max_queries]
    except Exception as exc:
        print(f"Query expansion fallback: {type(exc).__name__}")
        return [question]

def _to_text(value):
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        return "\n".join(_to_text(x) for x in value)
    if isinstance(value, dict):
        title = value.get("title") or value.get("content") or value.get("snippet") or ""
        url = value.get("url") or value.get("link") or ""
        return f"{title}\nURL: {url}\n{json.dumps(value, ensure_ascii=False)}"
    return str(value)

def parallel_retrieve(queries: List[str]) -> List[Dict[str, str]]:
    jobs = []
    with ThreadPoolExecutor(max_workers=8) as pool:
        for q in queries:
            jobs.extend([
                (q, "web", pool.submit(web_search_tool.invoke, {"query": q})),
                (q, "arxiv", pool.submit(arxiv_tool.invoke, q)),
                (q, "wikipedia", pool.submit(wiki_tool.invoke, q))
            ])
        results = []
        for q, source, future in jobs:
            try:
                text = _to_text(future.result(timeout=60))
                if text.strip():
                    results.append({"query": q, "source": source, "text": text})
            except Exception as exc:
                results.append({"query": q, "source": source,
                                "text": f"[SOURCE ERROR: {type(exc).__name__}: {exc}]"})
    return results

print("Parallel retrieval functions ready.")

Parallel retrieval functions ready.


## 6. Hybrid retrieval, deduplication, and reranking

We combine BM25 keyword matching with sentence-transformer semantic similarity. A cross-encoder then reranks the best candidates against the original question. Models are downloaded from Hugging Face on first use; no paid API key is needed.

In [6]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

# Multilingual embedding model helps with English/Arabic mixed research.
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker = CrossEncoder(RERANKER_MODEL_NAME)

def split_result_into_documents(results):
    docs = []
    for item in results:
        text = item["text"]
        # Keep source chunks manageable and retain source labels.
        chunks = [x.strip() for x in re.split(r"\n(?=\s*(?:\d+\.|Title:|URL:))", text) if x.strip()]
        if not chunks:
            chunks = [text]
        for chunk in chunks:
            if len(chunk) > 80 and not chunk.startswith("[SOURCE ERROR"):
                docs.append({"source": item["source"], "query": item["query"], "text": chunk[:5000]})
    return docs

def deduplicate_documents(docs):
    unique, seen = [], set()
    for doc in docs:
        normalized = re.sub(r"\W+", " ", doc["text"].lower()).strip()
        key = hashlib.sha256(normalized[:1200].encode("utf-8")).hexdigest()
        if key not in seen:
            seen.add(key)
            unique.append(doc)
    return unique

def hybrid_rerank(question: str, docs: List[Dict[str, str]], top_k: int = 10):
    docs = deduplicate_documents(docs)
    if not docs:
        return []
    tokenized = [re.findall(r"\w+", d["text"].lower()) for d in docs]
    bm25 = BM25Okapi(tokenized)
    bm25_scores = np.asarray(bm25.get_scores(re.findall(r"\w+", question.lower())), dtype=float)
    doc_embeddings = embedding_model.encode([d["text"] for d in docs], normalize_embeddings=True)
    query_embedding = embedding_model.encode([question], normalize_embeddings=True)
    semantic_scores = cosine_similarity(query_embedding, doc_embeddings)[0]
    # Normalize both retrieval scores before combining.
    def normalize(a):
        span = a.max() - a.min()
        return (a - a.min()) / span if span > 1e-9 else np.zeros_like(a)
    combined = 0.4 * normalize(bm25_scores) + 0.6 * normalize(semantic_scores)
    candidate_ids = np.argsort(combined)[::-1][:min(max(top_k * 3, top_k), len(docs))]
    candidates = [docs[i] for i in candidate_ids]
    pairs = [(question, d["text"]) for d in candidates]
    cross_scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, cross_scores), key=lambda x: float(x[1]), reverse=True)
    output = []
    for doc, score in ranked[:top_k]:
        output.append({**doc, "rerank_score": round(float(score), 4)})
    return output

print("Hybrid retrieval and reranking models loaded.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (12 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Hybrid retrieval and reranking models loaded.


## 7. Iterative retrieval — identify missing evidence and search again

After the first retrieval pass, the LLM identifies unanswered aspects and proposes a small number of follow-up searches. The final evidence pool is deduplicated and reranked again.

In [7]:
def identify_followup_queries(question: str, evidence: List[Dict[str, str]], max_followups: int = 2):
    evidence_text = "\n\n".join(
        f"[{d['source']}] {d['text'][:900]}" for d in evidence[:8]
    )
    prompt = f'''
Research question: {question}
Evidence collected so far:
{evidence_text}

Identify at most {max_followups} important unanswered aspects. Return ONLY a JSON array
of search queries. If evidence is sufficient, return [].
'''
    try:
        response = llm.invoke([SystemMessage(content="You are a critical research planner."),
                               HumanMessage(content=prompt)])
        raw = response.content if isinstance(response.content, str) else str(response.content)
        match = re.search(r"\[.*\]", raw, flags=re.S)
        return json.loads(match.group(0))[:max_followups] if match else []
    except Exception:
        return []

def run_enhanced_retrieval(question: str, top_k: int = 10):
    queries = expand_research_query(question)
    first_pass = parallel_retrieve(queries)
    docs = split_result_into_documents(first_pass)
    first_ranked = hybrid_rerank(question, docs, top_k=top_k)
    followups = identify_followup_queries(question, first_ranked)
    if followups:
        second_pass = parallel_retrieve(followups)
        docs.extend(split_result_into_documents(second_pass))
    final_ranked = hybrid_rerank(question, docs, top_k=top_k)
    return {"queries": queries, "followups": followups, "evidence": final_ranked}

print("Iterative retrieval pipeline ready.")

Iterative retrieval pipeline ready.


## 8. Expose the enhanced retrieval pipeline as an agent tool

This callable tool lets LangGraph use the complete multi-source search workflow as one capability. The three original sources remain independently callable as well.

In [8]:
from langchain_core.tools import tool

@tool
def deep_research_search(question: str) -> str:
    """Run expanded parallel web, ArXiv, and Wikipedia retrieval with hybrid ranking and reranking."""
    result = run_enhanced_retrieval(question, top_k=10)
    blocks = [f"Search queries: {result['queries']}",
              f"Follow-up queries: {result['followups']}", "Ranked evidence:"]
    for i, doc in enumerate(result["evidence"], 1):
        blocks.append(
            f"\n[{i}] Source={doc['source']} | rerank_score={doc['rerank_score']}\n{doc['text']}"
        )
    return "\n".join(blocks)[:24000]

print("Deep research retrieval tool ready.")

Deep research retrieval tool ready.


## 9. Fourth tool — YouTube Video Researcher

The tool validates YouTube URLs, extracts available captions in the requested/automatic language order, and preserves timestamps. If captions are unavailable, it returns a clear explanation instead of fabricating a transcript.

Supported requests include summarization, questions about the video, key moments, learning notes, and comparing multiple videos (the agent can call the tool for each URL).

In [9]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi


def extract_youtube_video_id(url: str) -> str:
    parsed = urlparse(url.strip())
    host = (parsed.hostname or "").lower()
    video_id = None
    if host in {"youtu.be", "www.youtu.be"}:
        video_id = parsed.path.strip("/").split("/")[0]
    elif host.endswith("youtube.com") or host.endswith("youtube-nocookie.com"):
        if parsed.path == "/watch":
            video_id = parse_qs(parsed.query).get("v", [None])[0]
        elif parsed.path.startswith(("/embed/", "/shorts/", "/live/")):
            parts = parsed.path.strip("/").split("/")
            video_id = parts[1] if len(parts) > 1 else None
    if not video_id or not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise ValueError("Invalid YouTube URL. Supply a standard public YouTube video URL.")
    return video_id


def fetch_youtube_transcript(url: str, languages=None):
    """Fetch existing YouTube captions. This does not download or transcribe audio."""
    video_id = extract_youtube_video_id(url)
    languages = languages or ["en", "ar"]
    api = YouTubeTranscriptApi()
    transcript = None
    language_code = "unknown"
    first_error = None

    # First try the preferred languages.
    try:
        transcript = api.fetch(video_id, languages=languages)
        language_code = getattr(transcript, "language_code", "unknown")
    except Exception as exc:
        first_error = exc

    # If preferred languages are absent, try the first available caption track.
    if transcript is None:
        try:
            available = list(api.list(video_id))
            if not available:
                raise RuntimeError("No caption tracks are available for this video.")
            chosen = available[0]
            transcript = chosen.fetch()
            language_code = getattr(chosen, "language_code", "unknown")
        except Exception as exc:
            detail = f"Preferred-language attempt: {type(first_error).__name__}: {first_error}. " if first_error else ""
            raise RuntimeError(
                detail + f"Could not retrieve captions ({type(exc).__name__}: {exc}). "
                "Possible causes: captions are disabled, the video is private/age-restricted, "
                "or YouTube is blocking/rate-limiting requests from Colab. "
                "If the error mentions RequestBlocked/IpBlocked, rerunning the cell may not help; "
                "try from a local computer or provide the transcript text manually."
            ) from exc

    snippets = getattr(transcript, "snippets", list(transcript))
    segments = []
    for item in snippets:
        start = float(getattr(item, "start", 0.0))
        text_value = str(getattr(item, "text", "")).replace("\n", " ").strip()
        if text_value:
            timestamp = time.strftime("%H:%M:%S", time.gmtime(start))
            segments.append({"timestamp": timestamp, "start_seconds": start, "text": text_value})
    if not segments:
        raise RuntimeError("The caption track was returned but contained no text.")
    return {"video_id": video_id, "language": language_code, "segments": segments}


@tool
def youtube_video_research(url: str, question: str = "Summarize this video and extract its key points.") -> str:
    """Fetch a YouTube video's available captions and return timestamped transcript text.
    The agent should use this transcript to answer the user's question. No video/audio claims
    are made beyond the available captions.
    """
    try:
        data = fetch_youtube_transcript(url)
    except Exception as exc:
        return f"YouTube transcript unavailable: {exc}"

    transcript_text = "\n".join(
        f"[{segment['timestamp']}] {segment['text']}" for segment in data["segments"]
    )
    max_chars = 24000
    if len(transcript_text) > max_chars:
        third = max_chars // 3
        middle = len(transcript_text) // 2
        transcript_text = (
            transcript_text[:third]
            + "\n[... middle section omitted because transcript is long ...]\n"
            + transcript_text[middle-third//2:middle+third//2]
            + "\n[... later section ...]\n"
            + transcript_text[-third:]
        )
    return (
        f"Video: https://www.youtube.com/watch?v={data['video_id']}\n"
        f"Transcript language: {data['language']}\n"
        f"User's question: {question}\n\n"
        "Timestamped captions (use these timestamps; do not invent any):\n"
        + transcript_text
    )


def analyze_youtube_video(url: str, question: str):
    """Direct YouTube workflow that bypasses agent tool-selection ambiguity."""
    data = fetch_youtube_transcript(url)
    transcript_text = "\n".join(
        f"[{s['timestamp']}] {s['text']}" for s in data["segments"]
    )
    max_chars = 30000
    if len(transcript_text) > max_chars:
        third = max_chars // 3
        mid = len(transcript_text) // 2
        transcript_text = (
            transcript_text[:third]
            + "\n[... middle section omitted ...]\n"
            + transcript_text[mid-third//2:mid+third//2]
            + "\n[... ending section ...]\n"
            + transcript_text[-third:]
        )
    prompt = f"""You are analyzing a YouTube video's captions. Use ONLY the transcript for claims about the video.
Answer the request clearly for a beginner. Provide a concise summary, explain difficult concepts simply,
and list five important moments with timestamps copied from the transcript. Do not invent timestamps.
If the transcript is incomplete or does not support five moments, say so.

Video URL: {url}
Transcript language: {data['language']}
Request: {question}

TIMESTAMPED TRANSCRIPT:
{transcript_text}
"""
    response = invoke_with_retry(llm, prompt, attempts=3)
    content = response.content if isinstance(response.content, str) else str(response.content)
    report = f"Transcript language: {data['language']}\nVideo ID: {data['video_id']}\n\n{content}"
    try:
        from IPython.display import display, Markdown
        display(Markdown(report))
    except Exception:
        print(report)
    return report

print("YouTube transcript tool ready. The tool no longer makes a nested LLM request.")

YouTube transcript tool ready. The tool no longer makes a nested LLM request.


## 10. Build the four-tool research agent

The agent has four primary tools: enhanced deep research, ArXiv, Wikipedia, and YouTube. The enhanced deep-research tool itself searches the web, ArXiv, and Wikipedia in parallel, applies hybrid retrieval/reranking, and performs a follow-up retrieval pass.

In [10]:
from langgraph.prebuilt import create_react_agent

tools = [
    deep_research_search,
    arxiv_tool,
    wiki_tool,
    youtube_video_research
]

SYSTEM_PROMPT = '''
You are a careful, helpful research assistant for both everyday users and advanced researchers.

TOOLS AND WHEN TO USE THEM:
- deep_research_search: Use for broad research questions, current information, multi-source evidence,
  query expansion, parallel web/ArXiv/Wikipedia retrieval, hybrid ranking, reranking, and follow-up search.
- arxiv_academic_search: Use when the user specifically wants academic papers or scientific preprints.
- wikipedia_search: Use for concise background, definitions, and established concepts.
- youtube_video_research: Use whenever the user provides a YouTube URL or asks about the content of a video.
  Pass both the URL and the user's actual question. Never claim to have watched video frames; this tool
  analyzes available captions/transcripts only.

RESEARCH RULES:
1. Understand the user's goal and select the appropriate tool(s).
2. For broad research, prefer deep_research_search rather than issuing many narrow searches manually.
3. For YouTube, distinguish transcript-supported facts from general background knowledge.
4. Never invent citations, video statements, transcript content, publication details, or timestamps.
5. Mention when evidence is missing, conflicting, old, or based on automatically generated captions.
6. Explain technical topics clearly for ordinary users unless the user requests technical depth.
7. Structure substantial reports with an executive summary, findings, evidence, limitations, and sources.
8. Cite URLs or source identifiers returned by tools where available.
'''

research_agent = create_react_agent(model=llm, tools=tools, prompt=SYSTEM_PROMPT)
print("Research agent initialized with:", [t.name for t in tools])

Research agent initialized with: ['deep_research_search', 'arxiv_academic_search', 'wikipedia_search', 'youtube_video_research']


/tmp/ipykernel_487/4201846100.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(model=llm, tools=tools, prompt=SYSTEM_PROMPT)


## 11. Run the research agent

In [11]:
from langchain_core.messages import HumanMessage
from langgraph.errors import GraphRecursionError
from IPython.display import display, Markdown

MAX_STEPS = 45

def run_research(query: str):
    print("=" * 75)
    print("RESEARCH QUESTION:", query)
    print("=" * 75)
    inputs = {"messages": [HumanMessage(content=query)]}
    config = {"recursion_limit": MAX_STEPS}
    final_response = ""
    seen = 0
    try:
        for chunk in research_agent.stream(inputs, config=config, stream_mode="values"):
            messages = chunk["messages"][seen:]
            seen = len(chunk["messages"])
            for msg in messages:
                if msg.type == "ai":
                    for call in getattr(msg, "tool_calls", []):
                        print(f"TOOL: {call['name']} | args: {call['args']}")
                    if msg.content and not getattr(msg, "tool_calls", []):
                        final_response = msg.content
                elif msg.type == "tool":
                    print(f"RESULT: {msg.name} | {len(str(msg.content))} characters")
    except GraphRecursionError:
        print("Agent reached the step limit. Try a narrower question.")
    if final_response:
        display(Markdown(final_response))
    else:
        print("No final response was produced.")
    return final_response

## 12. Test YouTube caption retrieval directly

This test checks caption retrieval independently of the LLM. If it fails with `RequestBlocked` or `IpBlocked`, YouTube is blocking Colab's network route; that is not an LLM/model error.

In [12]:
YOUTUBE_TEST_URL = "https://youtu.be/0QzopZ78w9M?si=80TZHMrNM6EJmeDS"

try:
    transcript_data = fetch_youtube_transcript(YOUTUBE_TEST_URL)
    print("Video ID:", transcript_data["video_id"])
    print("Transcript language:", transcript_data["language"])
    print("Number of transcript segments:", len(transcript_data["segments"]))
    print("First five segments:")
    for segment in transcript_data["segments"][:5]:
        print(f"[{segment['timestamp']}] {segment['text']}")
except Exception as exc:
    print("CAPTION RETRIEVAL FAILED:", exc)
    print("If the message says RequestBlocked/IpBlocked, try running locally or use a transcript export.")

Video ID: 0QzopZ78w9M
Transcript language: en
Number of transcript segments: 7
First five segments:
[00:00:00] Get transcript from YouTube video.
[00:00:02] First, open the YouTube app on your
[00:00:04] device and tap on the video for which
[00:00:06] you want to get the transcript. Now, at
[00:00:08] the description, tap on more.


## 13. Analyze the YouTube video (direct workflow)

This cell deliberately bypasses the ReAct agent for a deterministic test: fetch captions first, then send the timestamped transcript to the selected LLM. This separates YouTube caption errors from LLM/API errors.

In [13]:
youtube_url = "https://youtu.be/0QzopZ78w9M?si=80TZHMrNM6EJmeDS"
youtube_request = (
    "Summarize the video's main ideas, explain difficult concepts for a beginner, "
    "and list five important moments with timestamps. Use only timestamps present in the transcript."
)
youtube_report = analyze_youtube_video(youtube_url, youtube_request)

Transcript language: en
Video ID: 0QzopZ78w9M

**Video Summary (beginner‑friendly)**  
The clip is a quick, step‑by‑step guide on how to view a YouTube video’s automatic transcript directly from the YouTube app:

1. Open the YouTube app and select the video you’re interested in.  
2. Tap the **“More”** (⋮) button in the video description area.  
3. Choose **“Show transcript.”**  
4. The transcript appears on the screen; you can read it or copy it as needed.  

That’s the entire process—no extra software or settings are required.

---

### Simple Explanation of the Key Steps  

| Concept | What it means (in plain language) |
|---------|------------------------------------|
| **YouTube app** | The official YouTube program on your phone or tablet. |
| **Tap “More”** | Press the three‑dot menu (⋮) that shows extra options for the video. |
| **Show transcript** | A built‑in feature that displays the automatically generated text of everything spoken in the video. |
| **Transcript** | A written version of the spoken words, generated by YouTube’s speech‑recognition system. It can be useful for reading, searching, or copying the video’s content. |

No technical jargon is needed—just follow the on‑screen taps.

---

### Five Important Moments (with timestamps taken directly from the transcript)

| # | Timestamp | What Happens |
|---|-----------|--------------|
| 1 | **[00:00:02]** | The narrator tells you to open the YouTube app on your device. |
| 2 | **[00:00:04]** | You tap on the video for which you want the transcript. |
| 3 | **[00:00:06]** | In the description area, you tap the **“More”** (⋮) button. |
| 4 | **[00:00:08]** | You click on **“Show transcript.”** |
| 5 | **[00:00:11]** | The transcript appears on the screen, completing the process. |

These timestamps capture each distinct step of the tutorial.

## 14. Test broad research and the enhanced search pipeline

In [14]:
research_topic = "How have retrieval-augmented generation systems improved, and what are their limitations?"
research_report = run_research(research_topic)

RESEARCH QUESTION: How have retrieval-augmented generation systems improved, and what are their limitations?
TOOL: deep_research_search | args: {'question': 'retrieval-augmented generation improvements recent advances limitations'}
RESULT: deep_research_search | 24000 characters


**Retrieval‑Augmented Generation (RAG): How the field has progressed and where it still falls short**  

---

## Executive Summary  

| Aspect | Recent advances (2021‑2024) | Remaining limitations |
|--------|----------------------------|------------------------|
| **Retriever quality** | Dense‑vector models (e.g., DPR, ColBERT), hybrid sparse‑+‑dense search, graph‑based retrievers (GraphRAG), and efficient ANN indexes have dramatically raised recall and reduced latency【1†b14da0-00】. | Retrieval is still the *bottleneck*: missed, outdated, or noisy documents directly corrupt the generated answer【1†b14da0-00】【3†b14da0-00】. |
| **Dynamic / multi‑pass retrieval** | “RAG‑Token” and step‑wise retrieval let the model fetch new passages during decoding, improving multi‑hop reasoning【1†b14da0-00】. | Managing consistency across multiple retrieved chunks remains hard; contradictory evidence can cause hallucinations【1†b14da0-00】【2†79f246-01】. |
| **Fusion & reasoning** | Fusion‑in‑Decoder (FiD), EMDR2, G‑Retriever, recursive abstractive processing, and self‑reflective loops (SELF‑RAG, SelfMem) better integrate many passages and enable “self‑verification” of facts【2†8f0f49-01】【2†8f0f49-01】. | Fusion mechanisms add compute and can still produce shallow or incoherent synthesis, especially when passages are only loosely related【2†8f0f49-01】. |
| **Faithfulness & hallucination control** | RAGTruth benchmark, faithfulness‑aware decoding, and critique‑generate loops reduce factual errors【2†8f0f49-01】. | Hallucinations persist when retrieved text is ambiguous or when the model over‑relies on its parametric knowledge【1†b14da0-00】. |
| **Scalability & latency** | Approximate nearest‑neighbor (ANN) search, memory compression, and “lazy” graph‑RAG architectures enable sub‑second response times on billions of documents【4†ecdcb8-02】【4†ecdcb8-02】. | Real‑time low‑latency retrieval at production scale still demands heavy engineering (GPU‑accelerated indexes, sharding, caching)【4†ecdcb8-02】. |
| **Domain‑specific & up‑to‑date knowledge** | Plug‑and‑play document stores, fine‑tuned retrievers, and “knowledge‑injection” pipelines let RAG use proprietary or freshly crawled data without retraining the LLM【5†809d20-00】. | Keeping the external corpus fresh, bias‑free, and comprehensive is an operational challenge; stale or missing sources lead to incorrect answers, especially in high‑stakes domains like medicine【3†8ef1d6-02】. |
| **Evaluation** | New reference‑free metrics (RAGAS) and fine‑grained hallucination corpora (RAGTruth) give clearer signals for progress【1†b14da0-00】. | Benchmarks still focus on single‑turn QA; real‑world multi‑turn, reasoning‑heavy tasks are under‑represented. |

---

## Detailed Findings  

### 1. Core Architectural Improvements  

| Development | What changed | Impact |
|------------|--------------|--------|
| **Dense & hybrid retrievers** | Use of transformer‑based embeddings (e.g., DPR) combined with BM25 sparse scores. | ↑ Recall, ↓ latency; enables retrieval from massive corpora【1†b14da0-00】. |
| **Graph‑based retrieval (GraphRAG, Nebula GraphRAG, Fast GraphRAG)** | Documents are linked via entities/relations; retrieval can follow graph edges to gather multi‑hop evidence. | Better support for complex reasoning and knowledge‑graph queries【4†ecdcb8-02】. |
| **Dynamic (RAG‑Token) retrieval** | The model queries the retriever at each decoding step instead of a single static set. | Allows on‑the‑fly evidence gathering for long, multi‑step answers【1†b14da0-00】. |
| **Fusion‑in‑Decoder (FiD) & EMDR2** | Retrieved passages are encoded separately, then concatenated in the decoder with cross‑attention. | Improves handling of many passages (10‑20+) without overwhelming the encoder【1†b14da0-00】. |
| **Self‑reflective loops (SELF‑RAG, SelfMem)** | After an initial generation, the model critiques its answer, re‑retrieves if needed, and revises. | Reduces factual errors and improves faithfulness【2†8f0f49-01】. |
| **Hallucination‑aware decoding (RAGTruth, RAGAS)** | New evaluation suites and decoding constraints that penalize ungrounded tokens. | Provides concrete metrics for grounding; guides model design【2†8f0f49-01】. |

### 2. Performance Gains Observed  

* **Benchmark dominance** – On KILT, Natural Questions, TriviaQA, and HotpotQA, RAG‑style systems have become the top‑performing approaches since 2022, often beating pure LLM baselines by 5‑15 % absolute accuracy【3†324e28-00】.  
* **Long‑context LLMs** – The rise of models with 32 k‑64 k token windows (e.g., Claude‑2, Llama‑2‑70B‑Chat) lets RAG feed many retrieved passages without truncation, boosting multi‑document QA【4†ecdcb8-02】.  
* **Latency improvements** – Approximate nearest‑neighbor libraries (FAISS, ScaNN, Milvus) and “lazy” graph traversal reduce end‑to‑end latency to < 200 ms for 1 M‑scale corpora, making RAG viable for real‑time chatbots【4†ecdcb8-02】.  

### 3. Persistent Limitations  

| Category | Typical failure mode | Example evidence |
|----------|---------------------|------------------|
| **Retrieval quality** | Missed or outdated documents → answer contains stale facts. | Medical RAG model “LiVersa” cited obsolete guidelines because the latest version was absent from its corpus【3†8ef1d6-02】. |
| **Noise & irrelevance** | Retrieval returns partially relevant passages; model may blend contradictory statements. | “Collapse” phenomenon where the retriever repeatedly returns the same generic doc regardless of query, leading to generic or wrong answers【3†8ef1d6-02】. |
| **Fusion consistency** | Multiple passages provide overlapping but slightly different numbers; model picks one arbitrarily. | Studies note “contradictory evidence” as a key challenge for multi‑pass RAG【1†b14da0-00】. |
| **Scalability** | Indexing billions of documents and serving low‑latency ANN queries requires heavy infrastructure. | Production‑grade RAG still needs GPU‑accelerated indexes and careful sharding【4†ecdcb8-02】. |
| **Training‑inference gap** | Models are trained with a limited set of retrieved passages but at inference they query a much larger memory, causing distribution shift. | “Training‑inference gap” highlighted as an open research problem【2†79f246-01】. |
| **Domain‑specific bias** | If the external corpus is biased (e.g., over‑representation of certain viewpoints), the generated answer inherits that bias. | General limitation noted across many blogs and surveys【5†809d20-00】【6†401ba9-00】. |
| **Iterative reasoning** | Current RAG pipelines lack true multi‑turn reasoning; they cannot re‑evaluate earlier retrieved evidence. | “Lack of iterative reasoning” repeatedly cited as a limitation in recent surveys【6†401ba9-00】. |
| **Safety & high‑stakes use** | In medicine, law, finance, a single hallucinated fact can be dangerous. | Review of medical RAG stresses the need for rigorous validation before deployment【3†8ef1d6-02】. |

### 4. Emerging Research Directions  

1. **Hybrid retrieval‑generation training** – Jointly fine‑tune retriever and generator on the same objective to close the training‑inference gap (e.g., “Learning to filter context for RAG” 2023)【3†324e28-00】.  
2. **Graph‑augmented RAG** – Leveraging knowledge graphs to guide retrieval and enforce logical consistency (GraphRAG, Nebula GraphRAG)【4†ecdcb8-02】.  
3. **Adversarial noise injection** – Counter‑intuitively adding random documents can improve reasoning by forcing the model to be selective (Cuconasu et al., 2024)【2†8f0f49-01】.  
4. **Faithfulness‑aware decoding** – Integrating verification modules (e.g., external fact‑checkers) into the generation loop.  
5. **Continual corpus updating** – Automated pipelines that ingest fresh web data, re‑embed, and refresh ANN indexes with minimal downtime.  

---

## Evidence Overview  

| Source | Key points extracted |
|--------|----------------------|
| **Systematic Review (arXiv 2025)** – https://arxiv.org/html/2507.18910v1 | Summarizes progress in retrievers, multi‑pass RAG, fusion mechanisms, and outlines open challenges (retrieval quality, fusion contradictions, latency). |
| **Comprehensive Survey (arXiv 2025)** – https://arxiv.org/html/2506.00054v1 | Details faithfulness‑aware decoding (SELF‑RAG, SelfMem), hallucination benchmarks (RAGTruth), and noise‑mitigation strategies. |
| **Wikipedia entry** – https://en.wikipedia.org/wiki/Retrieval-augmented_generation | Provides baseline definition, historical context (first proposed 2020), and high‑level benefits (reducing hallucinations, up‑to‑date knowledge). |
| **EmergentMind blog** – https://www.emergentmind.com/topics/retrieval-augmented-generation | Highlights retrieval quality, scalability, and training‑inference gap as research frontiers. |
| **UnfoldAI article** – https://unfoldai.com/rag-limitations | Lists practical limitations: flawed data, inadequate augmentation, difficulty finding close matches. |
| **Towards Data Science** – https://towardsdatascience.com/the-limitations-and-advantages-of-retrieval-augmented-generation-rag-9ec9b4ae3729 | Discusses lack of iterative reasoning, data organization needs, and bias/out‑of‑date information. |
| **Microsoft Azure blog** – https://azure.microsoft.com/en-us/resources/cloud-computing-dictionary/what-is-retrieval-augmented-generation-rag | Explains architecture and the benefit of grounding LLMs in external knowledge. |
| **Medical mini‑review (PMC)** – https://pmc.ncbi.nlm.nih.gov/articles/PMC12059965 | Shows concrete medical‑domain failures when the knowledge base is stale or incomplete. |
| **RAGFlow 2024 year‑in‑review** – https://ragflow.io/blog/the-rise-and-evolution-of-rag-in-2024-a-year-in-review | Lists new open‑source tools (GraphRAG, LightRAG, HippoRAG) that target latency and graph‑based reasoning. |

---

## Practical Take‑aways  

* **If you are building a production RAG system:**  
  * Invest early in a high‑quality, regularly refreshed document store (vector + sparse indexes).  
  * Use a *dynamic* retrieval strategy (RAG‑Token) for tasks that need multi‑hop reasoning.  
  * Add a post‑generation verification step (self‑critique or external fact‑checker) to curb hallucinations.  

* **If you are evaluating RAG research:**  
  * Prefer benchmarks that measure grounding (RAGTruth, RAGAS) rather than pure QA accuracy.  
  * Report latency and index size alongside accuracy to reflect real‑world constraints.  

* **Open research problems worth tackling:**  
  * Unified training of retriever + generator to eliminate the training‑inference gap.  
  * Robust fusion mechanisms that can reconcile contradictory evidence without exploding compute.  
  * Scalable, low‑cost ANN pipelines for billions of documents (e.g., quantized indexes, hierarchical retrieval).  

---

**Bottom line:** Retrieval‑augmented generation has moved from a proof‑of‑concept to a mainstream architecture that dramatically improves factuality, up‑datability, and domain adaptability of LLMs. Yet the quality of the retrieved evidence, the efficiency of large‑scale retrieval, and the ability to reason over multiple pieces of information remain the most pressing hurdles. Continued work on joint retriever‑generator training, graph‑aware retrieval, and self‑verification will be key to unlocking truly reliable, knowledge‑intensive AI systems.

## 15. Compare the three LLM providers

Run the same prompt with each provider after adding its API key to Colab Secrets. This lightweight test compares basic response latency and successful invocation. For a fair quality comparison, use the same benchmark questions and manually score correctness, tool selection, evidence use, and completeness. Provider free quotas and model availability can change.

In [15]:
import time

def compare_llm_providers(prompt="Explain hybrid search in RAG in simple terms."):
    """Smoke-test the configured Groq model."""
    started = time.time()
    try:
        response = invoke_with_retry(llm, prompt, attempts=3)
        text = response.content if isinstance(response.content, str) else str(response.content)
        return {"provider": "Groq", "model": MODEL_NAME, "status": "success",
                "seconds": round(time.time() - started, 2), "preview": text[:500]}
    except Exception as exc:
        return {"provider": "Groq", "model": MODEL_NAME, "status": f"failed: {type(exc).__name__}",
                "seconds": round(time.time() - started, 2), "preview": str(exc)[:500]}

# Optional smoke test:
# print(compare_llm_providers())
